In [14]:
from snack_stack_graph import build_graph
from tools.config import get_llm
from tools.menu import MENU_COLLECTION_NAME, load_menu_documents
from tools.orders import load_orders_documents
from tools.vector_store import VectorStore

# Initialize the Graph

In [15]:
MENU_COLLECTION_NAME = "menu_collection"
orders = load_orders_documents("../../data/orders.json")
persist_directory = "../../data/chroma_store_3"
documents = load_menu_documents("../../data/menu.json")
vector_store = VectorStore(persist_directory=persist_directory, collection_name=MENU_COLLECTION_NAME)
menu_store = vector_store.get_create_collection(documents)

Loading weights: 100%|█████████████████████| 314/314 [00:00<00:00, 6053.97it/s]


In [16]:
llm = get_llm()

In [17]:
graph, context = build_graph(orders=orders, menu_collection=menu_store, llm=llm)

In [18]:
MENU_AGENT_NODE = "menu_agent_node"
MENU_AGENT_TOOL_NODE = "menu_agent_tool_node"
ORDER_AGENT_NODE = "order_agent_node"
ORDER_AGENT_TOOL_NODE = "order_agent_tool_node"
SYNTHESIZER_AGENT_NODE = "synthesizer_agent_node"
ORCHESTRATOR_AGENT_NODE = "orchestrator_agent"

def print_node_output_field(field_name:str, node_output:dict):
    if field_name in node_output:
        print(f"{field_name} {node_output[field_name]}\n")

def chat_with_user(prompt:str, user:str):
    input = {
        "user_input": prompt,
        "messages": [],
        "output": "",
        "route": ""
    }
    config={"configurable": {"thread_id": user}}
    for step in graph.stream(input, config, context=context):
        for node_name, node_output in step.items():
            print(f"\n--- Using: {node_name} ---")
            print(node_output)
            if node_name == ORCHESTRATOR_AGENT_NODE:
                for task in node_output['tasks']:
                    print(f"\n\ndispatching to agent {task.agent} with description: {task.description}")
                print_node_output_field('messages', node_output)
                print_node_output_field('requires_synthesis', node_output)
                print("\n----------------------------")
            elif node_name == MENU_AGENT_NODE:
                print_node_output_field('messages', node_output)
                print_node_output_field('menu_agent_output', node_output)
                print("\n----------------------------")
            elif node_name == MENU_AGENT_TOOL_NODE:
                print_node_output_field('messages', node_output)
                print("\n----------------------------")

                    
  #       stream_and_save_response(f"Route: {node_output["route"]}")
  #     elif node_name in [BLOG_WRITER_TOOL_NODE, SOCIAL_MEDIA_WRITER_NODE]:
  #       for msg in node_output.get("messages", []):
  #         stream_and_save_response(f"Tool Result: {str(msg.content)[:200000]}...")  
  #     else:
  #       stream_and_save_response(f"{node_output["output"]}")

In [19]:
user = "user_123"
chat_with_user("Can you recommend some italian food",user)


--- Using: orchestrator_agent ---
{'tasks': [AgentTask(agent='menu_agent', description='User is asking for recommendations on Italian food. Please provide Italian food recommendations from the menu.')], 'requires_synthesis': False, 'user_input': 'Can you recommend some italian food'}


dispatching to agent menu_agent with description: User is asking for recommendations on Italian food. Please provide Italian food recommendations from the menu.
requires_synthesis False


----------------------------
{'messages': [], 'user_input': 'Can you recommend some italian food', 'tasks': [AgentTask(agent='menu_agent', description='User is asking for recommendations on Italian food. Please provide Italian food recommendations from the menu.')], 'requires_synthesis': False}

--- Using: menu_agent_node ---
{'messages': [SystemMessage(content=' \nYou are a menu agent. Your job is to answer questions about our restaurants menu. \nYou can use the *search_menu_catalog* catalog to answer the questions re

/Users/elsadunsmoor/projects/snack-stack-ai/.venv/lib/python3.14/site-packages/pydantic/functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ContextSchema(orders=<mul...input at 0x12ec74040>)]), input_type=ContextSchema])
  function=lambda v, h: h(v), schema=original_schema
/Users/elsadunsmoor/projects/snack-stack-ai/.venv/lib/python3.14/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ContextSchema(orders=<mul...input at 0x12ec74040>)]), input_type=ContextSchema])
  return self.__pydantic_serializer__.to_python(



--- Using: menu_agent_node ---
{'menu_agent_output': 'Great! Here are my top 3 Italian food recommendations from our menu:\n\n1. **Margherita Pizza** - $299 (Rating: 4.7/5) ⭐\n   - Classic thin crust with tomato, mozzarella, and basil\n   - Vegetarian option\n\n2. **Aglio e Olio** - $279 (Rating: 4.5/5)\n   - Spaghetti with garlic, chilli, olive oil, and parsley\n   - Vegan option\n\n3. **Vegan Pasta Primavera** - $349 (Rating: 4.5/5)\n   - Penne with seasonal vegetables, olive oil, and garlic\n   - Vegan option\n\nAll three dishes are highly rated! The Margherita Pizza has the highest rating and is a classic Italian favorite. Would you like more information about any of these dishes, or would you like me to search for other Italian options?', 'messages': [AIMessage(content='Great! Here are my top 3 Italian food recommendations from our menu:\n\n1. **Margherita Pizza** - $299 (Rating: 4.7/5) ⭐\n   - Classic thin crust with tomato, mozzarella, and basil\n   - Vegetarian option\n\n2. **

In [20]:
menu_store.similarity_search("Margherita Pizza")

[Document(id='dcc1aa74-0c05-40b4-8822-87305926ef1c', metadata={'Dish': 'Margherita Pizza', 'Cuisine': 'Italian', 'Dietary': 'Veg'}, page_content='Dish: Margherita Pizza\n        Cuisine: Italian\n        Price: 299\n        Rating: 4.7\n        Dietary: Veg\n        Description: Classic thin crust with tomato, mozzarella, basil'),
 Document(id='e1ad8424-c5a8-46c7-9558-e595a1fdb16a', metadata={'Dietary': 'Vegan', 'Cuisine': 'Italian', 'Dish': 'Aglio e Olio'}, page_content='Dish: Aglio e Olio\n        Cuisine: Italian\n        Price: 279\n        Rating: 4.5\n        Dietary: Vegan\n        Description: Spaghetti with garlic, chilli, olive oil, parsley'),
 Document(id='8e5e7e18-d139-4034-af0c-36eaeb7d9dcd', metadata={'Dietary': 'Vegan', 'Cuisine': 'Italian', 'Dish': 'Vegan Pasta Primavera'}, page_content='Dish: Vegan Pasta Primavera\n        Cuisine: Italian\n        Price: 349\n        Rating: 4.5\n        Dietary: Vegan\n        Description: Penne with seasonal vegetables, olive oil, g

In [21]:
menu_store.get()

{'ids': ['dcc1aa74-0c05-40b4-8822-87305926ef1c',
  '8e5e7e18-d139-4034-af0c-36eaeb7d9dcd',
  '1c7f7612-1e79-40db-a47e-60aeb73be097',
  '599eef2e-28ac-4b3e-9b77-a6059b1a6f7e',
  'c1b995ad-a600-4658-b2c0-3e2164d3b54d',
  '12a6a38b-171a-4009-8c74-99c0be9d30f7',
  'e1ad8424-c5a8-46c7-9558-e595a1fdb16a',
  '176ba188-725b-4f6a-a69f-ca0fde12834a'],
 'embeddings': None,
 'documents': ['Dish: Margherita Pizza\n        Cuisine: Italian\n        Price: 299\n        Rating: 4.7\n        Dietary: Veg\n        Description: Classic thin crust with tomato, mozzarella, basil',
  'Dish: Vegan Pasta Primavera\n        Cuisine: Italian\n        Price: 349\n        Rating: 4.5\n        Dietary: Vegan\n        Description: Penne with seasonal vegetables, olive oil, garlic',
  'Dish: Butter Chicken\n        Cuisine: Indian\n        Price: 379\n        Rating: 4.9\n        Dietary: GF\n        Description: Creamy tomato curry with tender chicken and naan',
  'Dish: Vegan Buddha Bowl\n        Cuisine: Fusion\n

In [22]:
user = "user_123"
chat_with_user("what's the order status for order 124",user)


--- Using: orchestrator_agent ---
{'tasks': [AgentTask(agent='order_agent', description='Check the order status for order 124')], 'requires_synthesis': False, 'user_input': "what's the order status for order 124"}


dispatching to agent order_agent with description: Check the order status for order 124
requires_synthesis False


----------------------------

--- Using: order_agent_node ---
{'order_agent_output': [], 'messages': [AIMessage(content=[], additional_kwargs={}, response_metadata={'id': 'msg_011Ce3KZtfvaeniFqyeirzmf', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 1284, 'output_tokens': 3, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'cl